# Deep Learning Project : Classifiers

## Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import cv2
import os
import re

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.nn as nn
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from models.classifiers.NN.NN_1 import DeepFakeNN_1
from models.classifiers.CNN.ResNet34 import DeepFakeResNet34
from models.classifiers.CNN.Xception import DeepFakeXception
from models.classifiers.CNN.EfficientNet import DeepFakeEfficientNet
from models.classifiers.ViT.ViT_B_16 import DeepFakeViT
from ACPR.Project.utils.gradcam import visualize_gradcam_grid
import random

## Load Dataset

In [ ]:
real_images = glob("data/wiki/**/*.jpg", recursive=True)

fake_images_dict = {}
for folder in ["inpainting", "insight", "text2img"]:
    paths = glob(f"data/{folder}/**/*.jpg", recursive=True)
    fake_images_dict[folder] = paths

fake_images_dict["all_fakes"] = [
    path for folder in ["inpainting", "insight", "text2img"]
    for path in fake_images_dict[folder]
]

for folder, paths in fake_images_dict.items():
    print(f"  - {folder}: {len(paths)}")

In [ ]:
first_5_real = real_images[:5]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle("Real Images", fontsize=16)

for i, path in enumerate(first_5_real):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    axes[i].imshow(img_rgb)
    axes[i].set_title(f"Real {i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
for folder_name, paths in fake_images_dict.items():

    first_5_category = paths[:5]

    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    fig.suptitle(f"Fake Images: {folder_name.capitalize()}", fontsize=16)

    for i, path in enumerate(first_5_category):
        img = cv2.imread(path)

        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img_rgb)

        axes[i].set_title(f"{folder_name} {i+1}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

## Preprocessing

In [ ]:
class DeepFakeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'filepath']
        label = self.dataframe.loc[idx, 'label']

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = torch.tensor([label], dtype=torch.float32)

        return image, label

In [ ]:
def create_preprocessing_pipeline(real_list, fake_dict, batch_size=32, img_size=(224, 224),
                                   fake_index=3, model_type=None):
    """
    Takes a list of real images and a fake_dict, splits the data,
    and returns PyTorch DataLoaders and the split DataFrames.

    fake_index:
        0 = inpainting
        1 = insight
        2 = text2img
        3 = all_fakes (pre-built, balanced)

    model_type:
        'imagenet' -> Applies ImageNet mean/std normalization
        'xception' -> Applies Xception [-1, 1] normalization
        None       -> No extra normalization (use for custom CNNs)
    """

    print("Building DataFrames...")
    df_real = pd.DataFrame({'filepath': real_list, 'label': 0, 'source': 'wiki'})

    folder_names = list(fake_dict.keys())

    if not (0 <= fake_index <= len(folder_names) - 1):
        raise ValueError(f"fake_index must be between 0 and {len(folder_names) - 1}")

    selected = folder_names[fake_index]
    df_fake = pd.DataFrame({'filepath': fake_dict[selected], 'label': 1, 'source': selected})
    print(f"Using fake category: {selected} ({len(df_fake)} samples)")

    # Combine into full dataset
    df_full = pd.concat([df_real, df_fake], ignore_index=True)

    print("Splitting Data (70% Train, 15% Val, 15% Test)...")
    train_df, temp_df = train_test_split(
        df_full, test_size=0.30, random_state=42, stratify=df_full['label']
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
    )

    print("Setting up Transforms and DataLoaders...")
    transform_list: list = [
        transforms.Resize(img_size),
        transforms.ToTensor()
    ]

    if model_type == 'imagenet':
        print(" -> Applying ImageNet Normalization.")
        transform_list.append(
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        )
    elif model_type == 'xception':
        print(" -> Applying Xception [-1, 1] Normalization.")
        transform_list.append(
            transforms.Normalize(mean=[0.5, 0.5, 0.5],
                                 std=[0.5, 0.5, 0.5])
        )
    else:
        print(" -> No additional normalization applied.")

    data_transform = transforms.Compose(transform_list)

    train_dataset = DeepFakeDataset(train_df, transform=data_transform)
    val_dataset   = DeepFakeDataset(val_df,   transform=data_transform)
    test_dataset  = DeepFakeDataset(test_df,  transform=data_transform)

    use_pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=4, pin_memory=use_pin_memory)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=use_pin_memory)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=use_pin_memory)

    print("Pipeline Complete!\n")
    return train_loader, val_loader, test_loader, train_df, val_df, test_df

## Model Training

In [ ]:
def compute_binary_metrics(all_labels, all_preds, all_probs):
    accuracy = (np.array(all_preds) == np.array(all_labels)).mean() * 100
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        roc_auc = float("nan")

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [ ]:
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            if labels.dim() == 1:
                labels = labels.unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            all_labels.extend(labels.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())

    epoch_loss = running_loss / len(data_loader)
    metrics = compute_binary_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = epoch_loss

    return metrics


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, experiment_name, model_name, base_log_dir="experiment_logs", epochs=15, scheduler=None, patience=5, min_delta=1e-4):
    print(f"Starting experiment: {experiment_name}")

    history = {
        "train_loss": [], "train_accuracy": [], "train_precision": [],
        "train_recall": [], "train_f1": [], "train_roc_auc": [],
        "train_tp": [], "train_tn": [], "train_fp": [], "train_fn": [],
        "val_loss": [], "val_accuracy": [], "val_precision": [],
        "val_recall": [], "val_f1": [], "val_roc_auc": [],
        "val_tp": [], "val_tn": [], "val_fp": [], "val_fn": []
    }

    model_dir = os.path.join(base_log_dir, model_name)
    os.makedirs(model_dir, exist_ok=True)

    log_filepath = os.path.join(model_dir, "logs.txt")
    model_filepath = os.path.join(model_dir, f"model_{experiment_name}.pth")

    # Tracking variables for Early Stopping & Checkpointing
    best_val_loss = float('inf')
    epochs_no_improve = 0

    with open(log_filepath, "a") as f:
        f.write(f"Experiment: {experiment_name}\n")
        f.write("-" * 40 + "\n")
        f.flush()

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            all_labels, all_preds, all_probs = [], [], []

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                if labels.dim() == 1:
                    labels = labels.unsqueeze(1)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                probs = torch.sigmoid(outputs)
                preds = (probs >= 0.5).float()

                all_labels.extend(labels.detach().cpu().numpy().flatten())
                all_preds.extend(preds.detach().cpu().numpy().flatten())
                all_probs.extend(probs.detach().cpu().numpy().flatten())

            train_loss = running_loss / len(train_loader)
            train_metrics = compute_binary_metrics(all_labels, all_preds, all_probs)
            train_metrics["loss"] = train_loss

            val_metrics = evaluate_model(model, val_loader, criterion, device)

            history["train_loss"].append(train_metrics["loss"])
            history["train_accuracy"].append(train_metrics["accuracy"])
            history["train_precision"].append(train_metrics["precision"])
            history["train_recall"].append(train_metrics["recall"])
            history["train_f1"].append(train_metrics["f1"])
            history["train_roc_auc"].append(train_metrics["roc_auc"])
            history["train_tp"].append(train_metrics["tp"])
            history["train_tn"].append(train_metrics["tn"])
            history["train_fp"].append(train_metrics["fp"])
            history["train_fn"].append(train_metrics["fn"])

            history["val_loss"].append(val_metrics["loss"])
            history["val_accuracy"].append(val_metrics["accuracy"])
            history["val_precision"].append(val_metrics["precision"])
            history["val_recall"].append(val_metrics["recall"])
            history["val_f1"].append(val_metrics["f1"])
            history["val_roc_auc"].append(val_metrics["roc_auc"])
            history["val_tp"].append(val_metrics["tp"])
            history["val_tn"].append(val_metrics["tn"])
            history["val_fp"].append(val_metrics["fp"])
            history["val_fn"].append(val_metrics["fn"])

            log_entry = (
                f"Epoch {epoch+1}:\n"
                f"  TRAIN -> Loss {train_metrics['loss']:.4f}, Accuracy {train_metrics['accuracy']:.2f}%, Precision {train_metrics['precision']:.4f}, Recall {train_metrics['recall']:.4f}, F1 {train_metrics['f1']:.4f}, ROC-AUC {train_metrics['roc_auc']:.4f}, TP {train_metrics['tp']}, TN {train_metrics['tn']}, FP {train_metrics['fp']}, FN {train_metrics['fn']}\n"
                f"  VAL   -> Loss {val_metrics['loss']:.4f}, Accuracy {val_metrics['accuracy']:.2f}%, Precision {val_metrics['precision']:.4f}, Recall {val_metrics['recall']:.4f}, F1 {val_metrics['f1']:.4f}, ROC-AUC {val_metrics['roc_auc']:.4f}, TP {val_metrics['tp']}, TN {val_metrics['tn']}, FP {val_metrics['fp']}, FN {val_metrics['fn']}\n"
            )

            # --- Checkpointing and Early Stopping Logic ---
            current_val_loss = val_metrics["loss"]

            # The model must improve by at least min_delta to reset the patience counter
            if current_val_loss < (best_val_loss - min_delta):
                best_val_loss = current_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), model_filepath)
                log_entry += f"  [*] New best model saved! (Val Loss: {best_val_loss:.4f})\n"
            else:
                epochs_no_improve += 1
                log_entry += f"  [-] No significant improvement for {epochs_no_improve} epoch(s).\n"

            f.write(log_entry)
            f.flush()

            if scheduler is not None:
                scheduler.step()
                current_lr = scheduler.get_last_lr()[0]
            else:
                current_lr = optimizer.param_groups[0]["lr"]

            print(f"Finished {log_entry.strip()} | LR: {current_lr:.6f}")

            # Early Stopping Check
            if epochs_no_improve >= patience:
                stop_msg = f"\nEarly stopping triggered! Validation loss hasn't improved by at least {min_delta} in {patience} epochs."
                print(stop_msg)
                f.write(stop_msg + "\n")
                break

    # Restore best weights before returning
    print(f"\nLoading best model weights from: {model_filepath}")
    model.load_state_dict(torch.load(model_filepath, map_location=device))
    print(f"Logs saved to: {log_filepath}")

    return model, history

In [ ]:
def run_experiment_suite(model_factory, model_name, img_size, model_type, lr,
                         fake_images_dict, real_images, device, base_log_dir="experiment_logs", epochs=15):
    """
    Runs the 4 dataset experiments (inpainting, insight, text2img, all_fakes) for a given model.
    """
    all_models = {}
    all_histories = {}
    all_test_metrics = {}

    model_dir = os.path.join(base_log_dir, model_name)
    os.makedirs(model_dir, exist_ok=True)
    log_file = os.path.join(model_dir, "logs.txt")

    folder_names = list(fake_images_dict.keys())

    for fake_idx in range(4):
        experiment_name = "all_fakes" if fake_idx == 3 else folder_names[fake_idx]

        # Check if we should skip
        if os.path.exists(log_file):
            with open(log_file, "r") as f:
                if f"Experiment: {experiment_name}" in f.read():
                    print(f"Skipping {experiment_name.upper()}: Results already found in {log_file}")
                    continue

        print(f"\n{'='*60}")
        print(f"STARTING EXPERIMENT: {experiment_name.upper()} ({model_name})")
        print(f"{'='*60}")

        train_loader, val_loader, test_loader, train_df, _, _ = create_preprocessing_pipeline(
            real_list=real_images,
            fake_dict=fake_images_dict,
            batch_size=32,
            img_size=img_size,
            fake_index=fake_idx,
            model_type=model_type,
        )

        # --- Calculate dynamic pos_weight ---
        num_real = len(train_df[train_df['label'] == 0])
        num_fake = len(train_df[train_df['label'] == 1])

        weight_ratio = num_real / num_fake
        pos_weight = torch.tensor([weight_ratio], dtype=torch.float32).to(device)

        print(f"Data Balance -> Real: {num_real}, Fake: {num_fake} | Applying pos_weight: {weight_ratio:.4f}")
        # -------------------------------------------------

        # Model, Criterion, Optimizer, Scheduler
        model = model_factory().to(device)

        if device.type == 'cuda' and int(torch.__version__.split('.')[0]) >= 2:
            print("Optimizing model with torch.compile()... ")
            model = torch.compile(model)

        # Pass the calculated weight into the loss function!
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        # This safely handles both frozen and unfrozen models
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

        # Train
        trained_model, train_history = train_model(
            model=model, train_loader=train_loader, val_loader=val_loader,
            criterion=criterion, optimizer=optimizer, device=device,
            experiment_name=experiment_name, model_name=model_name,
            base_log_dir=base_log_dir, epochs=epochs, scheduler=scheduler
        )

        # Evaluate
        test_metrics = evaluate_model(trained_model, test_loader, criterion, device)

        # Log Test Metrics
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(
                f"  TEST  -> Loss {test_metrics['loss']:.4f}, Accuracy {test_metrics['accuracy']:.2f}%, "
                f"Precision {test_metrics['precision']:.4f}, Recall {test_metrics['recall']:.4f}, "
                f"F1 {test_metrics['f1']:.4f}, ROC-AUC {test_metrics['roc_auc']:.4f}, "
                f"TP {test_metrics['tp']}, TN {test_metrics['tn']}, FP {test_metrics['fp']}, FN {test_metrics['fn']}\n\n"
            )

        all_models[experiment_name] = trained_model
        all_histories[experiment_name] = train_history
        all_test_metrics[experiment_name] = test_metrics

    print(f"\n ALL 4 {model_name} EXPERIMENTS COMPLETED SUCCESSFULLY!")
    return all_models, all_histories, all_test_metrics

## Results Visualization Helpers

In [ ]:
def parse_experiment_logs(log_filepath):
    parsed_histories = {}
    parsed_test_metrics = {}
    current_experiment = None

    if not os.path.exists(log_filepath):
        print(f"Log file not found at {log_filepath}")
        return parsed_histories, parsed_test_metrics

    METRIC_FIELDS = ["train", "val"]
    HISTORY_KEYS  = ["loss", "accuracy", "precision", "recall", "f1", "roc_auc"]

    with open(log_filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line.startswith("Experiment:"):
                current_experiment = line.split("Experiment:")[1].strip()
                parsed_histories[current_experiment] = {
                    f"{split}_{metric}": []
                    for split in METRIC_FIELDS
                    for metric in HISTORY_KEYS
                }

            elif current_experiment:
                if   "TRAIN ->" in line: prefix = "train"
                elif "VAL   ->" in line: prefix = "val"
                elif "TEST  ->" in line: prefix = "test"
                else: continue

                patterns = {
                    "loss":      r"Loss ([\d\.]+)",
                    "accuracy":  r"Accuracy ([\d\.]+)",
                    "precision": r"Precision ([\d\.]+)",
                    "recall":    r"Recall ([\d\.]+)",
                    "f1":        r"F1 ([\d\.]+)",
                    "roc_auc":   r"ROC-AUC ([\d\.]+)",
                }
                extracted = {}
                for metric, pattern in patterns.items():
                    match = re.search(pattern, line)
                    if match:
                        value = float(match.group(1))
                        # Normalize accuracy from percentage to 0-1
                        extracted[metric] = value / 100.0 if metric == "accuracy" else value

                if prefix in ("train", "val"):
                    for metric, value in extracted.items():
                        key = f"{prefix}_{metric}"
                        if key in parsed_histories[current_experiment]:
                            parsed_histories[current_experiment][key].append(value)
                elif prefix == "test":
                    parsed_test_metrics[current_experiment] = extracted

    return parsed_histories, parsed_test_metrics

In [ ]:
def plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=None):
    if log_filepath:
        print(f"Fetching plot data directly from logs: {log_filepath}")
        histories_dict, test_metrics_dict = parse_experiment_logs(log_filepath)
    if not histories_dict:
        print("No history data available to plot.")
        return

    experiments = list(histories_dict.keys())
    palette     = plt.cm.tab10.colors
    exp_colors  = {name: palette[i % len(palette)] for i, name in enumerate(experiments)}

    METRICS = [
        ("loss",      "Loss",      "Loss"),
        ("accuracy",  "Accuracy",  "Accuracy (0-1)"),
        ("precision", "Precision", "Precision"),
        ("recall",    "Recall",    "Recall"),
        ("f1",        "F1 Score",  "F1"),
        ("roc_auc",   "ROC-AUC",   "ROC-AUC"),
    ]

    # Validate and report missing/empty metrics before plotting
    print("=" * 60)
    print("DATA VALIDATION REPORT")
    print("=" * 60)
    has_warnings = False
    for exp_name in experiments:
        history = histories_dict[exp_name]
        for split in ("train", "val"):
            for key, title, _ in METRICS:
                values = history.get(f"{split}_{key}", [])
                if not values:
                    print(f"  [MISSING]  {exp_name} | {split.upper():5s} | {title}")
                    has_warnings = True
                elif any(v is None or (isinstance(v, float) and (v != v)) for v in values):
                    print(f"  [NaN/None] {exp_name} | {split.upper():5s} | {title}")
                    has_warnings = True
                elif len(values) != len(history.get("train_loss", [])):
                    print(f"  [LENGTH MISMATCH] {exp_name} | {split.upper():5s} | {title} "
                          f" got {len(values)} entries vs {len(history.get('train_loss', []))} epochs")
                    has_warnings = True
        if test_metrics_dict:
            test = test_metrics_dict.get(exp_name, {})
            for key, title, _ in METRICS:
                if key == "loss":
                    continue
                if key not in test:
                    print(f"  [MISSING]  {exp_name} | TEST  | {title}")
                    has_warnings = True
    if not has_warnings:
        print("  All metrics present and valid for all experiments.")
    print("=" * 60 + "\n")

    # Build figure: 6 rows (metrics) 3 cols (Train | Val | Test)
    N_ROWS = len(METRICS)
    fig, axes = plt.subplots(
        N_ROWS, 3,
        figsize=(20, N_ROWS * 4),
        constrained_layout=True
    )
    fig.suptitle("Experiment Results\n\n", fontsize=18, fontweight="bold")

    # Column headers
    for col, label in enumerate(["Train", "Validation", "Test"]):
        axes[0, col].set_title(label, fontsize=13, fontweight="bold", pad=14)

    for row_idx, (key, title, ylabel) in enumerate(METRICS):
        ax_train = axes[row_idx, 0]
        ax_val   = axes[row_idx, 1]
        ax_test  = axes[row_idx, 2]

        # Train and Val line plots
        for split, ax in (("train", ax_train), ("val", ax_val)):
            for exp_name in experiments:
                values = histories_dict[exp_name].get(f"{split}_{key}", [])
                if not values:
                    continue
                epochs = range(1, len(values) + 1)
                ax.plot(epochs, values,
                        label=exp_name,
                        color=exp_colors[exp_name],
                        linewidth=1.8, marker='o', markersize=3)
            ax.set_ylabel(ylabel, fontsize=8)
            ax.set_xlabel("Epoch", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, linestyle=':', alpha=0.6)
            # Row label on the leftmost axis only
            if split == "train":
                ax.set_ylabel(f"{title}\n\n{ylabel}", fontsize=8)

        # Test bar chart
        if key != "loss":   # test loss is usually not reported, keep slot clean
            values = [
                (test_metrics_dict or {}).get(exp, {}).get(key, 0)
                for exp in experiments
            ]
            bars = ax_test.bar(
                range(len(experiments)), values,
                color=[exp_colors[e] for e in experiments],
                edgecolor="white", linewidth=0.8
            )
            for bar, v in zip(bars, values):
                if v > 0:
                    ax_test.annotate(
                        f"{v:.3f}",
                        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", va="bottom", fontsize=8, fontweight="bold"
                    )
            ax_test.set_xticks(list(range(len(experiments))))
            ax_test.set_xticklabels(experiments, rotation=25, ha="right", fontsize=7)
            ax_test.set_ylabel(ylabel, fontsize=8)
            ax_test.tick_params(axis='y', labelsize=7)
            ax_test.set_ylim(0, min(1.15, max(values, default=1) * 1.2) if any(values) else 1)
            ax_test.grid(True, axis='y', linestyle=':', alpha=0.6)
            if not any(values):
                ax_test.text(0.5, 0.5, "No Test Data", ha="center", va="center",
                             transform=ax_test.transAxes, fontsize=8, color="gray")
        else:
            ax_test.axis("off")
            ax_test.text(0.5, 0.5, "Test Loss\nNot Reported", ha="center", va="center",
                         transform=ax_test.transAxes, fontsize=8, color="gray")

    handles, labels = axes[0, 0].get_legend_handles_labels()

    axes[0, 1].legend(
        handles, labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.25),
        ncol=len(experiments),
        fontsize=10,
        frameon=True,
        title="Experiments",
        title_fontsize=10
    )

    plt.show()

In [ ]:
indices = random.sample(range(min(len(v) for v in fake_images_dict.values())), 10)

samples = [
    {
        "inpainting": fake_images_dict["inpainting"][idx],
        "insight":    fake_images_dict["insight"][idx],
        "text2img":   fake_images_dict["text2img"][idx],
        "all_fakes":  fake_images_dict["all_fakes"][idx],
    }
    for idx in indices
]

## Model Loader

In [ ]:
def load_models(model_factory, model_name, experiments, device, in_memory):
    """
    For each experiment, use the in-memory model if available, otherwise
    load the saved checkpoint from disk.
    """
    models = {}

    for exp_name in experiments:
        if in_memory:
            print(f"[{exp_name}] using in-memory model")
            model = in_memory[exp_name]
        else:
            ckpt_path = os.path.join("experiment_logs", model_name, f"model_{exp_name}.pth")
            if not os.path.exists(ckpt_path):
                raise FileNotFoundError(
                    f"[{exp_name}] no in-memory model and checkpoint not found at:\n"
                    f"  {ckpt_path}\n"
                    f"Train the model first or pass it via `in_memory`."
                )
            print(f"[{exp_name}] loading checkpoint from {ckpt_path}")
            model = model_factory()
            model.load_state_dict(torch.load(ckpt_path, map_location=device))

        model.to(device).eval()
        models[exp_name] = model

    return models

## Device Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Device set to: {device}")

### NN

#### NN_1

In [ ]:
img_size = (64, 64)

# --- NN_1 ---
all_models, all_histories, all_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeNN_1(img_size),
    model_name="NN_1",
    img_size=img_size,
    model_type=None,
    lr=0.001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
log_file_path = os.path.join("experiment_logs", "NN_1", "logs.txt")

# Check if both dictionaries exist in memory AND have data in them
if ('all_histories' in locals() and all_histories) and ('all_test_metrics' in locals() and all_test_metrics):
    plot_experiment_results(
        histories_dict=all_histories,
        test_metrics_dict=all_test_metrics,
        log_filepath=None
    )
else:
    plot_experiment_results(
        histories_dict=None,
        test_metrics_dict=None,
        log_filepath=log_file_path
    )

### CNN


#### ResNet

#### Fine Tuning Last Layer


In [ ]:
img_size = (224, 224)

# --- ResNet34 Frozen ---
all_resnet_frozen_models, all_resnet_frozen_histories, all_resnet_frozen_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeResNet34(freeze_features=True),
    model_name="ResNet34_frozen",
    img_size=img_size,
    model_type="imagenet",
    lr=0.001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
resnet_log_path = os.path.join("experiment_logs", "ResNet34_frozen", "logs.txt")

if ('all_resnet_frozen_histories' in locals() and all_resnet_frozen_histories) and ('all_resnet_frozen_test_metrics' in locals() and all_resnet_frozen_test_metrics):
    plot_experiment_results(histories_dict=all_resnet_frozen_histories, test_metrics_dict=all_resnet_frozen_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=resnet_log_path)

In [ ]:
models_resnet_frozen = load_models(lambda: DeepFakeResNet34(freeze_features=True), "ResNet34_frozen",
                                   ["inpainting", "insight", "text2img", "all_fakes"], device, all_resnet_frozen_models)

In [ ]:
for i, images in enumerate(samples):
    visualize_gradcam_grid(models_dict=models_resnet_frozen, images=images, device=device,
                           title=f"ResNet34 Frozen sample {i + 1}");

#### Fine Tuning All Layers

In [ ]:
img_size = (224, 224)

# --- ResNet34 Full ---
all_resnet_models, all_resnet_histories, all_resnet_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeResNet34(freeze_features=False),
    model_name="ResNet34_full",
    img_size=img_size,
    model_type="imagenet",
    lr=0.0001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
resnet_log_path = os.path.join("experiment_logs", "ResNet34_full", "logs.txt")

if ('all_resnet_histories' in locals() and all_resnet_histories) and ('all_resnet_test_metrics' in locals() and all_resnet_test_metrics):
    plot_experiment_results(histories_dict=all_resnet_histories, test_metrics_dict=all_resnet_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=resnet_log_path)

In [ ]:
models_resnet_full = load_models(lambda: DeepFakeResNet34(freeze_features=False), "ResNet34_full", ["inpainting", "insight", "text2img", "all_fakes"], device, all_resnet_models)

In [ ]:
for i, images in enumerate(samples):
    visualize_gradcam_grid(models_dict=models_resnet_full,   images=images, device=device, title=f"ResNet34 Full sample {i+1}");

### EfficientNet

#### Fine Tuning Last Layer

In [ ]:
img_size = (224, 224)

# --- EfficientNet Frozen ---
all_efficientnet_frozen_models, all_efficientnet_frozen_histories, all_efficientnet_frozen_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeEfficientNet(freeze_features=True),
    model_name="EfficientNet_frozen",
    img_size=img_size,
    model_type="imagenet",
    lr=0.001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
efficientnet_log_path = os.path.join("experiment_logs", "EfficientNet_frozen", "logs.txt")

if ('all_efficientnet_frozen_histories' in locals() and all_efficientnet_frozen_histories) and ('all_efficientnet_frozen_test_metrics' in locals() and all_efficientnet_frozen_test_metrics):
    plot_experiment_results(histories_dict=all_efficientnet_frozen_histories, test_metrics_dict=all_efficientnet_frozen_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=efficientnet_log_path)

In [ ]:
models_efficientnet_frozen = load_models(lambda: DeepFakeEfficientNet(freeze_features=True), "EfficientNet_frozen",
                                         ["inpainting", "insight", "text2img", "all_fakes"], device,
                                         all_efficientnet_frozen_models)

In [ ]:
for i, images in enumerate(samples):
    visualize_gradcam_grid(models_dict=models_efficientnet_frozen, images=images, device=device,
                           title=f"EfficientNet Frozen sample {i + 1}");

#### Fine Tunning All Layers

In [ ]:
img_size = (224, 224)

# --- EfficientNet Full ---
all_efficientnet_models, all_efficientnet_histories, all_efficientnet_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeEfficientNet(freeze_features=False),
    model_name="EfficientNet_full",
    img_size=img_size,
    model_type="imagenet",
    lr=0.0001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
efficientnet_log_path = os.path.join("experiment_logs", "EfficientNet_full", "logs.txt")

if ('all_efficientnet_histories' in locals() and all_efficientnet_histories) and ('all_efficientnet_test_metrics' in locals() and all_efficientnet_test_metrics):
    plot_experiment_results(histories_dict=all_efficientnet_histories, test_metrics_dict=all_efficientnet_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=efficientnet_log_path)

In [ ]:
models_efficientnet_full = load_models(lambda: DeepFakeEfficientNet(freeze_features=False), "EfficientNet_full", ["inpainting", "insight", "text2img", "all_fakes"], device, all_efficientnet_models)

In [ ]:
for i, images in enumerate(samples):
    visualize_gradcam_grid(models_dict=models_efficientnet_full,   images=images, device=device, title=f"EfficientNet Full sample {i+1}");

#### Xception

#### Fine Tuning Last Layer


In [ ]:
img_size = (299, 299)

# --- Xception Frozen ---
all_xception_frozen_models, all_xception_frozen_histories, all_xception_frozen_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeXception(freeze_features=True),
    model_name="Xception_frozen",
    img_size=img_size,
    model_type="xception",
    lr=0.001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
xception_log_path = os.path.join("experiment_logs", "Xception_frozen", "logs.txt")

if ('all_xception_frozen_histories' in locals() and all_xception_frozen_histories) and ('all_xception_frozen_test_metrics' in locals() and all_xception_frozen_test_metrics):
    plot_experiment_results(histories_dict=all_xception_frozen_histories, test_metrics_dict=all_xception_frozen_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=xception_log_path)

In [ ]:
models_xception_frozen = load_models(lambda: DeepFakeXception(freeze_features=True), "Xception_frozen",
                                     ["inpainting", "insight", "text2img", "all_fakes"], device,
                                     all_xception_frozen_models)

In [ ]:
for i, images in enumerate(samples):
    visualize_gradcam_grid(models_dict=models_xception_frozen, images=images, device=device,
                           title=f"Xception Frozen sample {i + 1}");

#### Fine Tuning All Layers

In [ ]:
img_size = (299, 299)

# --- Xception Full ---
all_xception_models, all_xception_histories, all_xception_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeXception(freeze_features=False),
    model_name="Xception_full",
    img_size=img_size,
    model_type="xception",
    lr=0.0001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
xception_log_path = os.path.join("experiment_logs", "Xception_full", "logs.txt")

if ('all_xception_histories' in locals() and all_xception_histories) and ('all_xception_test_metrics' in locals() and all_xception_test_metrics):
    plot_experiment_results(histories_dict=all_xception_histories, test_metrics_dict=all_xception_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=xception_log_path)

In [ ]:
models_xception_full = load_models(lambda: DeepFakeXception(freeze_features=False), "Xception_full", ["inpainting", "insight", "text2img", "all_fakes"], device, all_xception_models)

In [ ]:
for i, images in enumerate(samples):
    visualize_gradcam_grid(models_dict=models_xception_full,   images=images, device=device, title=f"Xception Full sample {i+1}");

## ViT

### ViT_B_16

#### Fine Tuning Last Layer

In [ ]:
img_size = (224, 224)

# --- ViT_B_16 Frozen ---
all_vit_frozen_models, all_vit_frozen_histories, all_vit_frozen_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeViT(freeze_features=True),
    model_name="ViT_B_16_frozen",
    img_size=img_size,
    model_type="imagenet",
    lr=0.001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
vit_frozen_log_path = os.path.join("experiment_logs", "ViT_frozen", "logs.txt")

if ('all_vit_frozen_histories' in locals() and all_vit_frozen_histories) and ('all_vit_frozen_test_metrics' in locals() and all_vit_frozen_test_metrics):
    plot_experiment_results(histories_dict=all_vit_frozen_histories, test_metrics_dict=all_vit_frozen_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=vit_frozen_log_path)

#### Fine Tuning All Layers

In [ ]:
img_size = (224, 224)

# --- ViT_B_16 Full ---
all_vit_full_models, all_vit_full_histories, all_vit_full_test_metrics = run_experiment_suite(
    model_factory=lambda: DeepFakeViT(freeze_features=False),
    model_name="ViT_B_16_full",
    img_size=img_size,
    model_type="imagenet",
    lr=0.0001,
    fake_images_dict=fake_images_dict,
    real_images=real_images,
    device=device,
    epochs=15
)

In [ ]:
vit_full_log_path = os.path.join("experiment_logs", "ViT_full", "logs.txt")

if ('all_vit_full_histories' in locals() and all_vit_full_histories) and ('all_vit_full_test_metrics' in locals() and all_vit_full_test_metrics):
    plot_experiment_results(histories_dict=all_vit_full_histories, test_metrics_dict=all_vit_full_test_metrics, log_filepath=None)
else:
    plot_experiment_results(histories_dict=None, test_metrics_dict=None, log_filepath=vit_full_log_path)